In [ ]:
import pandas as pd

# loading the CSV from the full path on E drive
df = pd.read_csv("E:/LogAnomalyDetector/data/parsed_logs.csv")
print(df)


In [ ]:
# Define anomaly logic
df['is_anomaly'] = df.apply(
    lambda row: 1 if row['level'] in ['ERROR', 'CRITICAL'] or pd.isna(row['ip']) else 0,
    axis=1
)

# Display updated dataframe
print(df)


In [ ]:
# Save the labeled data to a new CSV
df.to_csv("E:/LogAnomalyDetector/data/labeled_logs.csv", index=False)
print("✅ Labeled data saved to: data/labeled_logs.csv")


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import numpy as np

# Prepare features
df['has_ip'] = df['ip'].notna().astype(int)
df['level_encoded'] = df['level'].astype('category').cat.codes

X = df[['level_encoded', 'has_ip']]
y = df['is_anomaly']

# Split into train-test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the model
model = LogisticRegression()
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
print("📋 Classification Report:\n", classification_report(y_test, y_pred))


In [ ]:
import joblib

# Save the trained model
joblib.dump(model, "E:/LogAnomalyDetector/models/log_anomaly_model.pkl")

print("✅ Model saved to: models/log_anomaly_model.pkl")


In [ ]:
# Sample new logs 
new_logs = pd.DataFrame({
    'level': ['INFO', 'ERROR', 'CRITICAL', 'WARNING', 'INFO'],
    'ip': ['192.168.0.2', None, '192.168.0.10', '192.168.0.3', None]
})

# Preprocess e
new_logs['has_ip'] = new_logs['ip'].notna().astype(int)
new_logs['level_encoded'] = new_logs['level'].astype('category').cat.codes

# Load saved model
loaded_model = joblib.load("E:/LogAnomalyDetector/models/log_anomaly_model.pkl")

# Predict anomalies
X_new = new_logs[['level_encoded', 'has_ip']]
new_logs['is_anomaly_predicted'] = loaded_model.predict(X_new)

# Show the results
print(new_logs)


In [ ]:
# Save predictions to a CSV file
new_logs.to_csv("E:/LogAnomalyDetector/reports/predicted_logs.csv", index=False)
print("✅ Predicted log results saved to: reports/predicted_logs.csv")


In [ ]:
import matplotlib.pyplot as plt

# Count anomalies
counts = new_logs['is_anomaly_predicted'].value_counts().sort_index()
labels = ['Normal', 'Anomaly']

# Plot
plt.figure(figsize=(6, 4))
plt.bar(labels, counts, color=['green', 'red'])
plt.title("Predicted Anomaly Distribution")
plt.xlabel("Log Type")
plt.ylabel("Count")
plt.grid(axis='y')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd

# Load parsed real-time logs
df = pd.read_csv("data/parsed_live_logs.csv")

# Example basic anomaly condition: message contains 'error' or 'fail'
df['is_anomaly'] = df['message'].str.contains("error|fail|refused", case=False, na=False).astype(int)

print(df.head())


In [ ]:
import pandas as pd

# Load parsed logs
df = pd.read_csv("E:/LogAnomalyDetector/data/parsed_live_logs.csv")

# Simple labeling logic: any log without PID or from 'sendmail' is considered anomaly
df["is_anomaly"] = df.apply(
    lambda row: 1 if pd.isna(row["pid"]) or "sendmail" in row["service"].lower() else 0,
    axis=1
)

# Save the labeled file
df.to_csv("E:/LogAnomalyDetector/data/labeled_live_logs.csv", index=False)
print("Labeled data saved to: labeled_live_logs.csv")


In [ ]:
# === UPDATED TRAINING CELL: RandomForest on message_masked ===
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve

# ---------- CONFIG ----------
CSV_PATH = r"E:\LogAnomalyDetector\data\labeled_live_logs.csv"   # change if needed
MODELS_DIR = r"E:\LogAnomalyDetector\models"
MODEL_FILENAME = os.path.join(MODELS_DIR, "log_anomaly_model_masked.pkl")
VECT_FILENAME  = os.path.join(MODELS_DIR, "vectorizer_masked.pkl")

# ---------- Helpers ----------
def ensure_models_dir():
    os.makedirs(MODELS_DIR, exist_ok=True)

# ---------- 1) Load data ----------
print("Loading CSV:", CSV_PATH)
df = pd.read_csv(CSV_PATH)
print("Loaded rows:", len(df))
print("Columns:", list(df.columns))

# ---------- 2) Ensure required columns ----------
# If message_masked not present, try to create it using mask_message defined earlier
if "message_masked" not in df.columns:
    try:
        print("Column 'message_masked' not found -> trying to compute it using mask_message()")
        # mask_message must be defined earlier in your notebook (the improved masking function)
        df["message_masked"] = df["message"].apply(mask_message)
        print("Created 'message_masked'.")
    except NameError:
        raise RuntimeError("mask_message() not defined. Run the masking cell first, or add mask_message() to this notebook.")
    except Exception as e:
        raise RuntimeError("Failed to create message_masked: " + str(e))

# Ensure labels exist
if "is_anomaly" not in df.columns:
    raise RuntimeError("Column 'is_anomaly' not found. Ensure your CSV contains labels.")

# Quick class distribution
print("\nClass distribution (is_anomaly):")
print(df["is_anomaly"].value_counts())

# ---------- 3) Vectorize text ----------
vectorizer = TfidfVectorizer(max_df=0.95, min_df=1, ngram_range=(1,2), max_features=20000)
X = vectorizer.fit_transform(df["message_masked"].fillna("").astype(str))
y = df["is_anomaly"].astype(int).values

print("Vectorized shape:", X.shape)

# ---------- 4) Train / Test split (stratified if possible) ----------
if len(np.unique(y)) == 1:
    raise RuntimeError("Only one class present in y. Need at least two classes to train.")
    
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Train / Test sizes:", X_train.shape[0], "/", X_test.shape[0])

# ---------- 5) Train RandomForest with class_weight ----------
model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
print("Training RandomForest...")
model.fit(X_train, y_train)
print("Training done.")

# ---------- 6) Predict & Evaluate (hard preds) ----------
y_pred = model.predict(X_test)
print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred, zero_division=0))

print("\n=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred))

# ---------- 7) Suggested threshold via precision-recall curve ----------
# RandomForest gives predict_proba; get scores for positive class
if hasattr(model, "predict_proba"):
    y_scores = model.predict_proba(X_test)[:, 1]
    precisions, recalls, thresholds = precision_recall_curve(y_test, y_scores)
    # find threshold that gives precision >= 0.85 (if any)
    target_precision = 0.85
    idx = np.where(precisions >= target_precision)[0]
    if len(idx) > 0:
        # choose threshold with highest recall among those (practical choice)
        chosen = idx[np.argmax(recalls[idx])]
        suggested_threshold = thresholds[max(0, chosen-1)] if chosen>0 else thresholds[0]
        print("\nSuggested threshold for precision >= {:.2f} -> precision, recall, threshold: {:.3f}, {:.3f}, {:.4f}".format(
            target_precision, precisions[chosen], recalls[chosen], suggested_threshold
        ))
    else:
        # If no threshold meets the target, show best precision and its threshold
        best_prec_idx = np.argmax(precisions)
        # thresholds array shorter by 1 than precisions, so handle carefully
        th = thresholds[max(0, best_prec_idx-1)] if len(thresholds)>0 else 0.5
        print("\nNo threshold achieves precision >= {:.2f}. Best precision {:.3f} at approx threshold {:.4f}".format(
            target_precision, precisions[best_prec_idx], th
        ))
else:
    print("Model does not provide predict_proba; cannot compute precision-recall thresholds.")

# ---------- 8) Save model & vectorizer ----------
ensure_models_dir()
joblib.dump(model, MODEL_FILENAME)
joblib.dump(vectorizer, VECT_FILENAME)
print("\nModel and vectorizer saved to:", MODEL_FILENAME, ",", VECT_FILENAME)

# ---------- 9) Top text features (approx via RF feature_importances) ----------
try:
    feat_names = np.array(vectorizer.get_feature_names_out())
    importances = model.feature_importances_
    top_idx = np.argsort(importances)[::-1][:30]
    print("\nTop text features (feature : importance):")
    for i in top_idx:
        if importances[i] > 0:
            print(f"{feat_names[i]} : {importances[i]:.6f}")
except Exception as e:
    print("Could not display feature importances (reason):", e)

# ---------- Done ----------
print("\nFinished training run.")


In [ ]:
import os
import joblib

# Ensure directory exists
os.makedirs("../models", exist_ok=True)

# Save the model and vectorizer
joblib.dump(model, "../models/log_anomaly_model.pkl")
joblib.dump(vectorizer, "../models/vectorizer.pkl")

print("Model and vectorizer saved successfully.")


In [ ]:
import joblib

# Load the trained model and vectorizer
model = joblib.load("../models/log_anomaly_model.pkl")
vectorizer = joblib.load("../models/vectorizer.pkl")

print("Model and vectorizer loaded for inference.")


In [ ]:
import os
print(os.getcwd())


In [ ]:
import pandas as pd

# Load new parsed logs
df_new_logs = pd.read_csv("data/parsed_live_logs.csv")

# Vectorize the 'message' column
X_new = vectorizer.transform(df_new_logs["message"])

# Predict using the trained model
predictions = model.predict(X_new)

# Add predictions to the DataFrame
df_new_logs["is_anomaly_predicted"] = predictions

# Save the predictions
df_new_logs.to_csv("data/predicted_live_logs.csv", index=False)

print(" Prediction completed. Results saved to: data/predicted_live_logs.csv")


In [ ]:
import pandas as pd

df = pd.read_csv("data/predicted_live_logs.csv")
df.head(10)  # Show first 10 rows


In [ ]:
import pandas as pd

# Load your labeled dataset
df = pd.read_csv("data/labeled_live_logs.csv")

# Check class distribution
print("Class distribution:\n", df["is_anomaly"].value_counts())

# Quick look at data
df.head()


In [ ]:
# === DATA AUDIT ===
import pandas as pd
pd.set_option("display.max_colwidth", 200)

path = "data/labeled_live_logs.csv"   # adjust if your file name is different
df = pd.read_csv(path)

print("1) Data shape:", df.shape)
print("\n2) Columns:", df.columns.tolist())
print("\n3) Null counts per column:\n", df.isnull().sum())
print("\n4) Label distribution (is_anomaly):\n", df['is_anomaly'].value_counts(dropna=False))
print("\n5) Label proportions:\n", (df['is_anomaly'].value_counts(normalize=True).round(3)))
print("\n6) Sample anomalies (first 10 rows):")
display(df[df['is_anomaly']==1].head(10))
print("\n7) Sample normal logs (first 10 rows):")
display(df[df['is_anomaly']==0].head(10))


In [ ]:
# === Baseline Training: RandomForest with class_weight='balanced' ===
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve
from scipy.sparse import hstack
import joblib

pd.set_option("display.max_colwidth", 240)

# 1) Load labeled data
df = pd.read_csv("data/labeled_live_logs.csv")

# 2) Ensure masked text exists (fallback to raw message if not)
if 'message_masked' not in df.columns:
    df['message_masked'] = df['message'].astype(str)

# 3) Clean labels
if 'is_anomaly' not in df.columns:
    raise KeyError("Column 'is_anomaly' not found in data/labeled_live_logs.csv")
df = df.dropna(subset=['is_anomaly']).copy()
df['is_anomaly'] = df['is_anomaly'].astype(int)

# 4) TF-IDF vectorization (text)
tf = TfidfVectorizer(max_features=2000, ngram_range=(1,2), min_df=2, stop_words='english')
X_text = tf.fit_transform(df['message_masked'].astype(str))

# 5) Meta numeric features (ensure columns exist)
meta_cols = ['hour','weekday','host_total']
for c in meta_cols:
    if c not in df.columns:
        df[c] = 0
X_meta = df[meta_cols].fillna(-1).astype(float).values

# 6) Combine features
X = hstack([X_text, X_meta])
y = df['is_anomaly'].values

# 7) Train/test split (stratify if possible)
if len(np.unique(y)) == 1:
    raise ValueError("Only one class present in 'is_anomaly'. Need both classes to train supervised model.")
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

# 8) Train RandomForest with class_weight to help imbalance
clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', n_jobs=-1, random_state=42)
clf.fit(X_train, y_train)

# 9) Predict & evaluate
y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:,1]

print("=== Classification Report ===")
print(classification_report(y_test, y_pred, digits=4))
print("\n=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred))

# 10) Suggest a threshold for high precision (if available)
prec, rec, thr = precision_recall_curve(y_test, y_proba)
max_prec = max(prec) if len(prec)>0 else 0.0
print("\nMax precision achievable on test set (no thresholding):", round(max_prec,4))
# look for threshold that achieves precision >= 0.85 (teacher wants low FP). Adjust 0.85 if you prefer.
candidates = [(p,r,t) for p,r,t in zip(prec,rec,list(thr)+[1.0]) if p >= 0.85]
if candidates:
    chosen = max(candidates, key=lambda x: x[1])  # choose candidate with max recall among those with precision>=0.85
    print("Suggested threshold for precision>=0.85 -> precision, recall, threshold:", (round(chosen[0],4), round(chosen[1],4), chosen[2]))
else:
    print("No threshold in this run reached precision >= 0.85. You may lower the target or try calibration/ensemble methods.")

# 11) Save model and vectorizer
joblib.dump(clf, "models/log_anomaly_model.pkl")
joblib.dump(tf, "models/vectorizer.pkl")
print("\nModel and vectorizer saved to: models/log_anomaly_model.pkl , models/vectorizer.pkl")

# 12) Show top text features by importance (for explainability)
try:
    importances = clf.feature_importances_
    n_text = X_text.shape[1]
    text_importances = importances[:n_text]
    top_n = 15
    top_idx = np.argsort(text_importances)[-top_n:][::-1]
    feature_names = tf.get_feature_names_out()
    print("\nTop text features (feature_name : importance):")
    for i in top_idx:
        print(f"{feature_names[i]} : {round(text_importances[i], 6)}")
except Exception as e:
    print("\nCould not compute feature importances for text features:", e)


In [ ]:
# Step 1: improved masking function (paste and run)
import re
import pandas as pd

def mask_message(msg):
    """Mask common sensitive tokens in a log message and return masked string."""
    if pd.isna(msg):
        return ""
    s = str(msg)

    # IPv4 addresses -> <IP>
    s = re.sub(r'\b\d{1,3}(?:\.\d{1,3}){3}\b', '<IP>', s)

    # Long hex / alphanumeric tokens (>=8 chars) -> <ID>
    s = re.sub(r'\b[a-fA-F0-9]{8,}\b', '<ID>', s)
    s = re.sub(r'\b(?=[A-Za-z0-9]{8,})(?=.*[A-Za-z])(?=.*\d)[A-Za-z0-9]+\b', '<ID>', s)

    # PIDs / numeric ids -> <PID> (handles (1234), pid=1234, uid=1234)
    s = re.sub(r'\(\d+\)', '(<PID>)', s)
    s = re.sub(r'\b(pid|uid)=\d+\b', r'\1=<PID>', s, flags=re.IGNORECASE)
    s = re.sub(r'\bPID[:=]?\s*\d+\b', 'PID=<PID>', s, flags=re.IGNORECASE)

    # ISO-like timestamps -> <TS>
    s = re.sub(
        r'\b\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}(?:\.\d+)?(?:Z|[+-]\d{2}:?\d{2})?\b',
        '<TS>', s)

    # File paths (basic linux/windows patterns) -> <PATH>
    s = re.sub(r'(/[A-Za-z0-9_\-./]+)+', '<PATH>', s)
    s = re.sub(r'[A-Za-z]:\\[^\s]+', '<PATH>', s)

    # Collapse repeated whitespace
    s = re.sub(r'\s+', ' ', s).strip()
    return s


In [ ]:
# Test masking on your df
# (if df is not defined, first run: df = pd.read_csv("data/labeled_live_logs.csv"))
df["message_masked"] = df["message"].apply(mask_message)
display(df[["message", "message_masked", "is_anomaly"]].head(8))


In [ ]:
# 1-step: ensure message_masked exists and save CSV for training
import os
import pandas as pd
import re

print("Working dir:", os.getcwd())
print("Files in data/:", os.listdir("data") if os.path.isdir("data") else "no data folder")

# load the labeled file you used previously
csv_path = "data/labeled_live_logs.csv"   # change this path only if your file is elsewhere
if not os.path.exists(csv_path):
    raise FileNotFoundError(f"{csv_path} not found. Check path or run parsing step first.")

df = pd.read_csv(csv_path)
print("\nLoaded CSV rows:", len(df))
print("Columns:", df.columns.tolist())

# If message_masked already exists, show first rows and exit (so we don't overwrite)
if "message_masked" in df.columns:
    print("\nColumn 'message_masked' already present. Showing sample:")
    display(df[["message","message_masked","is_anomaly"]].head(8))
else:
    # Define the improved masking function (same one you used earlier)
    def mask_message(msg):
        if pd.isna(msg):
            return ""
        s = str(msg)

        # IPv4
        s = re.sub(r'\b\d{1,3}(?:\.\d{1,3}){3}\b', '<IP>', s)

        # Hex or long alnum IDs (like hashes / random ids) -> <ID>
        s = re.sub(r'\b(?:0x)?[A-Fa-f0-9]{8,}\b', '<ID>', s)
        s = re.sub(r'\b[A-Za-z0-9]{24,}\b', '<ID>', s)

        # PIDs: numbers in brackets or 'pid=1234'
        s = re.sub(r'\[(\d+)\]', '[<PID>]', s)
        s = re.sub(r'\bpid=\d+\b', 'pid=<PID>', s)
        s = re.sub(r'\bpids?\b[:=]?\d+\b', 'pid=<PID>', s)

        # ISO-like timestamps
        s = re.sub(r'\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}(?:\.\d+)?Z?', '<TS>', s)
        s = re.sub(r'\d{2}:\d{2}:\d{2}', '<TS>', s)

        # File paths (basic linux/windows patterns) -> <PATH>
        s = re.sub(r'(/[A-Za-z0-9_\-./]+)+', '<PATH>', s)
        s = re.sub(r'[A-Za-z]:\\[^ \n\r\t]+', '<PATH>', s)

        # Emails, hostnames
        s = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b', '<EMAIL>', s)
        s = re.sub(r'\b(?:[A-Za-z0-9-]+\.)+[A-Za-z]{2,}\b', lambda m: ('<HOST>' if '.' in m.group(0) else m.group(0)), s)

        # Collapse repeated whitespace
        s = re.sub(r'\s+', ' ', s).strip()

        return s

    # Create the masked column
    df["message_masked"] = df["message"].apply(mask_message)

    # Show some samples to verify
    print("\nCreated 'message_masked' - showing first 8 rows:")
    display(df[["message","message_masked","is_anomaly"]].head(8))

    # Save back to CSV (overwrite so training cell can use it)
    df.to_csv(csv_path, index=False)
    print("\nSaved updated CSV with 'message_masked' ->", csv_path)


In [ ]:
# ===== Stratified CV evaluation (copy & run this cell) =====
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score, roc_auc_score

# --- set path to your labeled CSV (adjust if needed) ---
csv_path = r"E:\LogAnomalyDetector\data\labeled_live_logs.csv"

# --- load CSV ---
df = pd.read_csv(csv_path)
print("Loading CSV:", csv_path)
print("Loaded rows:", len(df))
print("Columns:", list(df.columns))

# --- choose text column: prefer masked if available ---
text_col = "message_masked" if "message_masked" in df.columns else "message"
print("Using text column:", text_col)

# --- sanity checks ---
if "is_anomaly" not in df.columns:
    raise ValueError("Column 'is_anomaly' not found in dataset. Ensure labels exist.")

# prepare X and y
X_text = df[text_col].fillna("").astype(str)
y = df["is_anomaly"].astype(int)

# check minority class count for valid StratifiedKFold
min_class_count = int(y.value_counts().min())
if min_class_count < 2:
    raise ValueError(f"Not enough samples in minority class for stratified CV (min_count={min_class_count}). "
                     "Label more anomalies or use a different evaluation strategy.")

n_splits = min(5, min_class_count)  # no. of folds
print(f"Using StratifiedKFold with n_splits = {n_splits} (min class count = {min_class_count})")

# --- vectorize text (TF-IDF) ---
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X = vectorizer.fit_transform(X_text)
print("Vectorized shape:", X.shape)

# --- classifier (you can swap to your model) ---
clf = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1)

# --- Stratified CV loop ---
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

precisions = []
recalls = []
f1s = []
avg_precisions = []
roc_aucs = []

print("\nRunning CV folds:")
for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    # probability/score for PR/AUC if available
    if hasattr(clf, "predict_proba"):
        y_score = clf.predict_proba(X_test)[:, 1]
    else:
        # fall back to decision_function if available
        try:
            y_score = clf.decision_function(X_test)
        except Exception:
            y_score = y_pred  # fallback, not ideal for PR/AUC

    p = precision_score(y_test, y_pred, zero_division=0)
    r = recall_score(y_test, y_pred, zero_division=0)
    f = f1_score(y_test, y_pred, zero_division=0)

    # average_precision (PR AUC) / ROC AUC only if both classes present in test fold
    if len(np.unique(y_test)) > 1:
        ap = average_precision_score(y_test, y_score)
        try:
            roc = roc_auc_score(y_test, y_score)
        except Exception:
            roc = float("nan")
    else:
        ap = float("nan")
        roc = float("nan")

    precisions.append(p); recalls.append(r); f1s.append(f)
    avg_precisions.append(ap); roc_aucs.append(roc)

    print(f" Fold {fold:02d} -> precision={p:.3f}, recall={r:.3f}, f1={f:.3f}, AP={'{:.3f}'.format(ap) if not np.isnan(ap) else 'nan'}, ROC={'{:.3f}'.format(roc) if not np.isnan(roc) else 'nan'}")

# --- summary across folds ---
def mean_std_str(arr):
    arr = np.array(arr, dtype=float)
    return f"{np.nanmean(arr):.3f} ± {np.nanstd(arr):.3f}"

print("\nCV summary (mean ± std):")
print(" Precision (pos class):", mean_std_str(precisions))
print(" Recall    (pos class):", mean_std_str(recalls))
print(" F1-score  (pos class):", mean_std_str(f1s))
print(" PR AUC (avg precision):", mean_std_str(avg_precisions))
print(" ROC AUC:", mean_std_str(roc_aucs))

# Save vectorizer and a final model trained on full dataset (optional)
import joblib, os
os.makedirs("models", exist_ok=True)
clf.fit(X, y)
joblib.dump(clf, "models/log_anomaly_model_cv.pkl")
joblib.dump(vectorizer, "models/vectorizer_cv.pkl")
print("\nSaved final model+vectorizer to models/ (log_anomaly_model_cv.pkl, vectorizer_cv.pkl)")


In [ ]:
import pandas as pd

# Load the ADFA dataset (adjust path depending on where you are running Jupyter)
df = pd.read_csv("E:/LogAnomalyDetector/data/adfa_parsed.csv")   # Windows path
# df = pd.read_csv("/home/kali/logproject_workspace/data/adfa_parsed.csv")  # Kali path

# Show quick overview
print("Shape:", df.shape)   # rows x columns
print("\nColumns:", df.columns.tolist())
df.head()


In [ ]:
print("Class balance (is_anomaly):")
print(df['is_anomaly'].value_counts())

print("\nSplit distribution:")
print(df['split'].value_counts())


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Features (X) and labels (y)
X = df['message']
y = df['is_anomaly']

# Convert text to TF-IDF features
vectorizer = TfidfVectorizer(max_features=5000)  # you can tune max_features
X_vec = vectorizer.fit_transform(X)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_vec, y, test_size=0.3, random_state=42, stratify=y
)

# Train RandomForest
clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)

# Predictions
y_pred = clf.predict(X_test)

# Evaluation
print(classification_report(y_test, y_pred, digits=4))


In [ ]:
# Step 1: locate any ADFA csv files in the project workspace
import os, glob

root = os.getcwd()          # where the notebook server thinks we are
print("Notebook working dir:", root)
search_patterns = [
    "data/**/adfa*.csv",
    "data/**/adfa_parsed*.csv",
    "data/**/adfa*.csv",
    "**/adfa_parsed*.csv",
    "**/adfa*.csv"
]

found = []
for patt in search_patterns:
    # glob with recursive
    matches = glob.glob(os.path.join(root, patt), recursive=True)
    for m in matches:
        found.append(os.path.abspath(m))

found = sorted(set(found))
print(f"\nFound {len(found)} candidate files:")
for p in found:
    print(" -", p)

# if nothing found, also try searching one level up (common when mounted to E:)
if not found:
    alt_root = os.path.dirname(root)  # parent
    print("\nNo candidates found under notebook cwd. Searching parent dir:", alt_root)
    for patt in ["**/adfa*.csv", "**/*adfa*.csv"]:
        matches = glob.glob(os.path.join(alt_root, patt), recursive=True)
        for m in matches:
            found.append(os.path.abspath(m))
    found = sorted(set(found))
    print(f"\nFound {len(found)} candidate files (under parent):")
    for p in found:
        print(" -", p)

# helpful hint for you
if not found:
    print("\nIf nothing is found, run the parse script in whichever environment (Windows/Kali) produced the CSV and note the output path.")
else:
    print("\nChoose the most appropriate path above and assign it to `csv_path` in the next step.")


In [ ]:
import pandas as pd, os

# Step 2: point to the correct parsed ADFA file
csv_path = r"E:\LogAnomalyDetector\data\adfa_parsed.csv"  # Windows absolute path
print("Using CSV:", csv_path, "exists?", os.path.exists(csv_path))

# Load ADFA dataset
df_adfa = pd.read_csv(csv_path)

# Quick summary
print("\nShape:", df_adfa.shape)
print("Columns:", df_adfa.columns.tolist())
print("\nFirst 5 rows:")
display(df_adfa.head())


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Convert numeric syscall sequences into strings (for TF-IDF)
df_adfa["msg_str"] = df_adfa["message"].astype(str)

# Use TF-IDF to represent syscall sequences
vectorizer = TfidfVectorizer(max_features=5000, token_pattern=r"(?u)\b\d+\b")  
X = vectorizer.fit_transform(df_adfa["msg_str"])
y = df_adfa["is_anomaly"]

print("Feature matrix shape:", X.shape)
print("Label distribution:\n", y.value_counts())


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Split dataset (stratified so class balance is preserved)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train size:", X_train.shape, "Test size:", X_test.shape)

# Train Random Forest
model = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Evaluation
print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred))

print("\n=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred))


In [ ]:
# Quick dependency check
try:
    import pandas
    import numpy
    import sklearn
    import joblib
    import matplotlib
    import flask
    import plotly
    import imblearn
    import notebook
    print("✅ All dependencies are installed and ready!")
except ImportError as e:
    print("❌ Missing package:", e.name)


In [ ]:
import sys, subprocess
print("Notebook Python executable:", sys.executable)
print("Platform:", sys.platform)
# Check if flask is present in this env:
try:
    import flask
    print("Flask is already installed in this kernel.")
except ImportError:
    print("Flask is NOT installed in this kernel.")


In [ ]:
import sys
!{sys.executable} -m pip install flask


In [ ]:
import flask
print("✅ Flask installed successfully, environment ready!")


In [ ]:
# ========== STEP 1: Backup + Quick data audit ==========
import os, shutil, datetime
import pandas as pd
from collections import Counter

# --- adjust filename if your labeled file has a different name ---
DATA_PATH = "data/labeled_live_logs.csv"

# 1) backup (timestamped)
bk_dir = os.path.join("backups", datetime.datetime.now().strftime("%Y%m%d_%H%M%S"))
os.makedirs(bk_dir, exist_ok=True)
if os.path.exists(DATA_PATH):
    shutil.copy2(DATA_PATH, os.path.join(bk_dir, os.path.basename(DATA_PATH)))
    print(f"Backup created: {bk_dir}/{os.path.basename(DATA_PATH)}")
else:
    raise FileNotFoundError(f"Data file not found at: {DATA_PATH}")

# 2) load
df = pd.read_csv(DATA_PATH)
print("\n=== Loaded dataset ===")
print("Path:", DATA_PATH)
print("Shape:", df.shape)

# 3) columns & nulls
print("\nColumns:")
print(df.columns.tolist())
print("\nNulls per column:")
print(df.isnull().sum())

# 4) label column detection & distribution (common names)
label_candidates = [c for c in df.columns if c.lower() in ("is_anomaly","label","target","class","y")]
if label_candidates:
    label_col = label_candidates[0]
else:
    # fallback: assume last column is label if it has only 0/1
    label_col = df.columns[-1]
    if not set(df[label_col].dropna().unique()).issubset({0,1}):
        raise KeyError("Could not detect label column automatically. Please tell me the label column name.")
print(f"\nAssumed label column: '{label_col}'")

print("\nLabel distribution (counts):")
print(df[label_col].value_counts(dropna=False))

print("\nLabel proportions:")
print((df[label_col].value_counts(normalize=True, dropna=False)).round(4))

# 5) show head + sample anomalies
print("\nFirst 8 rows:")
display(df.head(8))

print("\nUp to 10 anomaly samples (if any):")
display(df[df[label_col]==1].head(10))

# 6) basic duplicates and timestamp detection
dups = df.duplicated().sum()
print(f"\nExact duplicate rows: {dups}")

ts_col = None
for c in df.columns:
    if 'time' in c.lower() or 'timestamp' in c.lower() or 'date' in c.lower():
        ts_col = c
        break
if ts_col:
    print(f"Found timestamp-like column: {ts_col}")
    try:
        df[ts_col] = pd.to_datetime(df[ts_col], errors='coerce')
        print("Timestamp parsing: min =", df[ts_col].min(), ", max =", df[ts_col].max())
    except Exception as e:
        print("Timestamp parse error:", e)
else:
    print("No timestamp-like column auto-detected.")


In [ ]:
# ========== STEP 2: Cleaning & base feature engineering ==========

# 1) Drop duplicates
df_clean = df.drop_duplicates().copy()
print(f"After dropping duplicates: {df.shape[0]} -> {df_clean.shape[0]} rows")

# 2) Ensure timestamp is datetime
df_clean['timestamp'] = pd.to_datetime(df_clean['timestamp'], errors='coerce')
print("Timestamp dtype:", df_clean['timestamp'].dtype)

# 3) Extract time features
df_clean['hour'] = df_clean['timestamp'].dt.hour
df_clean['weekday'] = df_clean['timestamp'].dt.weekday
df_clean['is_weekend'] = (df_clean['weekday'] >= 5).astype(int)

# 4) Regex-based flags on message
import re
df_clean['has_error'] = df_clean['message'].str.contains("error", case=False, na=False).astype(int)
df_clean['has_failed'] = df_clean['message'].str.contains("fail", case=False, na=False).astype(int)
df_clean['has_critical'] = df_clean['message'].str.contains("critical", case=False, na=False).astype(int)
df_clean['digit_count'] = df_clean['message'].str.count(r'\d')
df_clean['path_count'] = df_clean['message'].str.count(r'\/')

# 5) Host/Service grouping (rare values -> OTHER)
for col in ['host','service']:
    counts = df_clean[col].value_counts()
    rare = counts[counts < 2].index  # threshold = 2 (adjust if needed)


In [ ]:
# ========== STEP 3: TF-IDF text + combine engineered features ==========

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
import numpy as np

# Label
y = df_clean['is_anomaly']

# --- Features ---
# Text column
text_col = 'message_masked'

# Categorical cols (after rare grouping)
cat_cols = ['host','service']

# Numeric cols (engineered + pid)
num_cols = ['pid','hour','weekday','is_weekend','has_error','has_failed','has_critical','digit_count','path_count']

# TF-IDF vectorizer for message
tfidf = TfidfVectorizer(max_features=500, ngram_range=(1,2))

# Preprocessor: text + cat (OHE) + numeric (scaled)
preprocessor = ColumnTransformer(
    transformers=[
        ('text', tfidf, text_col),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', StandardScaler(), num_cols)
    ]
)

# Build the design matrix
X = preprocessor.fit_transform(df_clean)

print("Final feature matrix shape:", X.shape)
print("Label distribution:", np.bincount(y))


In [ ]:
# Fix: make y a positional NumPy array so y[test_idx] uses positional indexing
import numpy as np
y = np.asarray(y).reshape(-1)   # ensure 1-D numpy array
print("y shape:", y.shape, " - values:", np.bincount(y))


In [ ]:
# ========== STEP 4: Multi-Model Comparison (safe k-fold for small data) ==========
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.svm import LinearSVC
from sklearn.metrics import precision_score, recall_score, f1_score, make_scorer

# ensure X, y exist
X.shape, y.shape

# choose n_splits based on anomaly count (must be <= count of smallest class)
n_anom = int(np.sum(y == 1))
n_splits = min(max(2, n_anom), 4)   # at least 2, at most 4 (your data has 4 anomalies)
print(f"Anomalies: {n_anom}, using StratifiedKFold with n_splits={n_splits}\n")

kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

models = {
    "LogisticRegression": LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42),
    "LinearSVM": LinearSVC(class_weight='balanced', max_iter=20000, random_state=42),
}

scoring = {
    'precision': make_scorer(precision_score, pos_label=1, zero_division=0),
    'recall': make_scorer(recall_score, pos_label=1, zero_division=0),
    'f1': make_scorer(f1_score, pos_label=1, zero_division=0)
}

results = {}

# Supervised models
for name, model in models.items():
    scores = cross_validate(model, X, y, cv=kf, scoring=scoring, error_score='raise', n_jobs=-1)
    results[name] = {
        'precision_mean': scores['test_precision'].mean(),
        'precision_std': scores['test_precision'].std(),
        'recall_mean': scores['test_recall'].mean(),
        'recall_std': scores['test_recall'].std(),
        'f1_mean': scores['test_f1'].mean(),
        'f1_std': scores['test_f1'].std()
    }

# IsolationForest (unsupervised)
iso = IsolationForest(n_estimators=200, contamination=(n_anom / len(y)), random_state=42)
precisions, recalls, f1s = [], [], []
for train_idx, test_idx in kf.split(X, y):
    iso.fit(X[train_idx])
    y_pred_raw = iso.predict(X[test_idx])         # -1 anomaly, 1 normal
    y_pred = np.where(y_pred_raw == -1, 1, 0)     # map to 1=anomaly,0=normal
    y_true = y[test_idx]
    precisions.append(precision_score(y_true, y_pred, pos_label=1, zero_division=0))
    recalls.append(recall_score(y_true, y_pred, pos_label=1, zero_division=0))
    f1s.append(f1_score(y_true, y_pred, pos_label=1, zero_division=0))

results['IsolationForest'] = {
    'precision_mean': np.mean(precisions),
    'precision_std': np.std(precisions),
    'recall_mean': np.mean(recalls),
    'recall_std': np.std(recalls),
    'f1_mean': np.mean(f1s),
    'f1_std': np.std(f1s)
}

# Print nicely
print("=== Model comparison (averaged across folds) ===")
for name, m in results.items():
    print(f"{name:16s} Precision: {m['precision_mean']:.2f} ± {m['precision_std']:.2f} | "
          f"Recall: {m['recall_mean']:.2f} ± {m['recall_std']:.2f} | "
          f"F1: {m['f1_mean']:.2f} ± {m['f1_std']:.2f}")


In [ ]:
# ========== STEP 5: Final training and saving ==========
from sklearn.pipeline import Pipeline
import joblib

# Full pipeline = preprocessing + classifier
final_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42))
])

# Train on full dataset
final_model.fit(df_clean, y)

# Save model
joblib.dump(final_model, "models/best_model.joblib")

print(" Final Random Forest model trained and saved as models/best_model.joblib")


In [ ]:
# Reload and test
loaded_model = joblib.load("models/best_model.joblib")

# Take a small sample
sample = df_clean.sample(3, random_state=1)

preds = loaded_model.predict(sample)
print("Predictions on sample:", preds)
print("Ground truth:", sample['is_anomaly'].values)


In [ ]:
import matplotlib.pyplot as plt

# Get predicted probabilities for all samples
y_probs = final_model.predict_proba(df_clean)[:, 1]  # probability of anomaly class (1)

# Plot histogram of probabilities
plt.hist(y_probs, bins=20, edgecolor='k')
plt.xlabel("Anomaly probability")
plt.ylabel("Count of logs")
plt.title("Distribution of anomaly probabilities")
plt.show()

# Print some stats
print("Min prob:", y_probs.min())
print("Max prob:", y_probs.max())
print("Mean prob:", y_probs.mean())
print("Median prob:", np.median(y_probs))
